# 店舗売上需要予測: 実験 #006 特売リード＆ラグ特徴量 ＋ 3大GBDT多様性アンサンブル (LGB + CatBoost + XGBoost with GPU)

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **目的:** 時系列予測の最大の武器「既知の未来情報（テスト期間の特売情報）」を活用した **「特売リード＆ラグ特徴量」** ＋ **「3大GBDT多様性アンサンブル (RTX 4060 / Kaggle GPU)」** ＋ **「ゼロ売上フィルター」**
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **現在自己ベスト:** 実験 #005: **LB 0.43078 (210位)** 🏆

---

## 1. 特売プロモーション（`onpromotion`）の先行・後行効果

スーパーマーケットの需要予測において、「特売」は売上に劇的な影響を与えます：
1. **即時効果 (Contemporary):** 当日の特売により売上が跳ね上がる（既存 `onpromotion`）。
2. **事後影響・反動減 (Lag Effect):** 特売で買いだめ（Stockpiling）された直後は、家庭内に在庫があるため需要が急減する（`promo_lag_1`, `promo_lag_2`, `promo_lag_7`）。
3. **先行指標・買い控え (Lead Effect):** チラシ等で事前予告された特売（明日・明後日）があると、消費者は直前の買い控えを起こす（`promo_lead_1`, `promo_lead_2`）。
4. **モメンタム (Diff):** 特売が前日より急増した初日バースト効果（`promo_diff_1`）。

> 💡 **なぜ Lead（未来）を使ってリークにならないのか？**
> 売上実績 `sales` は未来の値をカンニングできませんが、**`test.csv` には 16 日間すべての `onpromotion`（特売品数）があらかじめ与えられています。**
> 実務でも「来週の折り込みチラシや特売予定」は店舗側が事前に決定しているため、合法的に使える最も強力な武器（外生変数）です！

## 2. ライブラリ読み込み & データ読み込み

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

# データディレクトリ自動検出
def find_data_dir():
    candidates = [
        '/kaggle/input/store-sales-time-series-forecasting',
        '/kaggle/input/competitions/store-sales-time-series-forecasting',
        '003_store-sales/data',
        '../data',
        'data'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    matches = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    matches_local = glob.glob('**/train.csv', recursive=True)
    if matches_local:
        return os.path.dirname(matches_local[0])
    if os.path.exists('/kaggle/input'):
        input_contents = os.listdir('/kaggle/input')
        raise FileNotFoundError(f'/kaggle/input にデータがありません。現在の中身: {input_contents}。右上の [+ Add Input] から "store-sales-time-series-forecasting" を追加してください。')
    raise FileNotFoundError('train.csv が見つかりませんでした。')

data_dir = find_data_dir()
print(f'データ読み込み元: {data_dir}')

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sub_df = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

# 直近21日間に売上ゼロのペアを特定（ゼロ売上フィルター用）
recent_train = train_df[train_df['date'] >= '2017-07-25']
recent_sum = recent_train.groupby(['store_nbr', 'family'])['sales'].sum().reset_index()
zero_pairs = set(zip(recent_sum[recent_sum['sales'] == 0]['store_nbr'], recent_sum[recent_sum['sales'] == 0]['family']))
print(f'直近21日間で売上ゼロのペア数: {len(zero_pairs)} / 1,782 ペア')

## 3. カレンダー ＆ 前年同期ラグ ＆ 特売リード・ラグ特徴量エンジニアリング

In [ ]:
# 全体結合
test_df['sales'] = np.nan
all_df = pd.concat([train_df, test_df], ignore_index=True)
all_df['log_sales'] = np.log1p(all_df['sales'])

# カレンダー特徴量
all_df['year'] = all_df['date'].dt.year
all_df['month'] = all_df['date'].dt.month
all_df['day'] = all_df['date'].dt.day
all_df['dayofweek'] = all_df['date'].dt.dayofweek
all_df['is_weekend'] = all_df['dayofweek'].isin([5, 6]).astype(int)
all_df['dayofyear'] = all_df['date'].dt.dayofyear
all_df['weekofyear'] = all_df['date'].dt.isocalendar().week.astype(int)
all_df['is_month_start'] = all_df['date'].dt.is_month_start.astype(int)
all_df['is_month_end'] = all_df['date'].dt.is_month_end.astype(int)
all_df['is_wage_day'] = ((all_df['day'] == 15) | (all_df['is_month_end'] == 1)).astype(int)
all_df['is_wage_day_after'] = ((all_df['day'] == 16) | (all_df['is_month_start'] == 1)).astype(int)

# 系列ごとに整列
all_df = all_df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
group_sales = all_df.groupby(['store_nbr', 'family'])['log_sales']
group_promo = all_df.groupby(['store_nbr', 'family'])['onpromotion']

print('時系列ラグ & ローリング平均（売上実績: Lag >= 16）...')
for lag in [16, 21, 28, 35, 364]:
    all_df[f'sales_lag_{lag}'] = group_sales.shift(lag)

all_df['sales_lag_364'] = all_df['sales_lag_364'].fillna(all_df['sales_lag_28'])

lag16 = group_sales.shift(16)
all_df['sales_rolling_mean_7'] = lag16.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
all_df['sales_rolling_mean_14'] = lag16.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(14, min_periods=1).mean()
)
all_df['sales_rolling_mean_28'] = lag16.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(28, min_periods=1).mean()
)

print('特売（onpromotion）先行（Lead）・後行（Lag）特徴量構築...')
# 事後影響ラグ（昨日・一昨日・前週同曜日の特売反動減）
all_df['promo_lag_1'] = group_promo.shift(1).fillna(0)
all_df['promo_lag_2'] = group_promo.shift(2).fillna(0)
all_df['promo_lag_7'] = group_promo.shift(7).fillna(0)

# 先行指標リード（明日・明後日の特売買い控え）
all_df['promo_lead_1'] = group_promo.shift(-1).bfill().fillna(0)
all_df['promo_lead_2'] = group_promo.shift(-2).bfill().fillna(0)

# モメンタム & 7日移動平均
all_df['promo_diff_1'] = all_df['onpromotion'] - all_df['promo_lag_1']
all_df['promo_rolling_mean_7'] = group_promo.transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)

train_feat = all_df[all_df['date'] <= '2017-08-15'].copy()
test_feat = all_df[all_df['date'] >= '2017-08-16'].copy()

features = [
    'store_nbr', 'onpromotion', 'year', 'month', 'day', 'dayofweek', 'is_weekend',
    'dayofyear', 'weekofyear', 'is_month_start', 'is_month_end', 'is_wage_day', 'is_wage_day_after',
    'sales_lag_16', 'sales_lag_21', 'sales_lag_28', 'sales_lag_35', 'sales_lag_364',
    'sales_rolling_mean_7', 'sales_rolling_mean_14', 'sales_rolling_mean_28',
    'promo_lag_1', 'promo_lag_2', 'promo_lag_7',
    'promo_lead_1', 'promo_lead_2',
    'promo_diff_1', 'promo_rolling_mean_7'
]
print(f'投入特徴量数: {len(features)} 列')

## 4. 33商品ファミリー別 3大GBDT多様性アンサンブル学習 (RTX 4060 / Kaggle GPU)

In [ ]:
val_start_date = '2017-07-31'
val_end_date = '2017-08-15'
train_start_date = '2016-01-01'

trn_mask = (train_feat['date'] >= train_start_date) & (train_feat['date'] < val_start_date)
val_mask = (train_feat['date'] >= val_start_date) & (train_feat['date'] <= val_end_date)

# 手元検証用直近0売上ペア
pre_val = train_df[(train_df['date'] >= '2017-07-10') & (train_df['date'] < val_start_date)]
zero_pre_val = pre_val.groupby(['store_nbr', 'family'])['sales'].sum().reset_index()
val_zero_pairs = set(zip(zero_pre_val[zero_pre_val['sales'] == 0]['store_nbr'], zero_pre_val[zero_pre_val['sales'] == 0]['family']))

families = train_df['family'].unique()
print(f'33商品ファミリーごとに 3大GBDT (LightGBM + CatBoost GPU + XGBoost GPU) を学習中...')

val_ens_preds = []
val_trues = []
test_predictions = []

# GPU利用確認
has_gpu = False
try:
    test_cb = cb.CatBoostRegressor(iterations=1, task_type='GPU', verbose=0)
    test_cb.fit(np.random.randn(10, 2), np.random.randn(10))
    has_gpu = True
    print('⚡ NVIDIA GPU (CUDA) 加速が有効です！')
except Exception as e:
    print(f'ℹ️ GPU未検出のためCPUモードで実行します: {e}')

cb_task_type = 'GPU' if has_gpu else 'CPU'
xgb_device = 'cuda' if has_gpu else 'cpu'

for fam in families:
    fam_trn_mask = trn_mask & (train_feat['family'] == fam)
    fam_val_mask = val_mask & (train_feat['family'] == fam)
    fam_tst_mask = (test_feat['family'] == fam)

    X_trn = train_feat.loc[fam_trn_mask, features].copy()
    y_trn = np.log1p(train_feat.loc[fam_trn_mask, 'sales'].values)
    X_v = train_feat.loc[fam_val_mask, features].copy()
    y_v = train_feat.loc[fam_val_mask, 'sales'].values
    X_tst = test_feat.loc[fam_tst_mask, features].copy()

    # 1. LightGBM (Leaf-wise)
    lgb_model = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        learning_rate=0.05,
        num_leaves=31,
        feature_fraction=0.8,
        random_state=42,
        verbose=-1,
        n_estimators=500
    )
    X_trn_lgb, X_v_lgb, X_tst_lgb = X_trn.copy(), X_v.copy(), X_tst.copy()
    X_trn_lgb['store_nbr'] = X_trn_lgb['store_nbr'].astype('category')
    X_v_lgb['store_nbr'] = X_v_lgb['store_nbr'].astype('category')
    X_tst_lgb['store_nbr'] = X_tst_lgb['store_nbr'].astype('category')

    lgb_model.fit(
        X_trn_lgb, y_trn,
        eval_set=[(X_v_lgb, np.log1p(y_v))],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    lgb_pred_v = lgb_model.predict(X_v_lgb)
    lgb_pred_tst = lgb_model.predict(X_tst_lgb)

    # 2. CatBoost (Symmetric Trees GPU)
    cb_model = cb.CatBoostRegressor(
        iterations=500,
        learning_rate=0.06,
        depth=6,
        loss_function='RMSE',
        task_type=cb_task_type,
        random_seed=42,
        verbose=0,
        early_stopping_rounds=30
    )
    cb_model.fit(X_trn, y_trn, eval_set=(X_v, np.log1p(y_v)), verbose=0)
    cb_pred_v = cb_model.predict(X_v)
    cb_pred_tst = cb_model.predict(X_tst)

    # 3. XGBoost (Depth-wise GPU)
    xgb_model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        device=xgb_device,
        random_state=42,
        early_stopping_rounds=30
    )
    xgb_model.fit(X_trn, y_trn, eval_set=[(X_v, np.log1p(y_v))], verbose=False)
    xgb_pred_v = xgb_model.predict(X_v)
    xgb_pred_tst = xgb_model.predict(X_tst)

    # 3大GBDTアンサンブル (50% LGB + 30% CB + 20% XGB)
    ens_pred_v = np.clip(np.expm1(0.50 * lgb_pred_v + 0.30 * cb_pred_v + 0.20 * xgb_pred_v), 0, None)
    ens_pred_tst = np.clip(np.expm1(0.50 * lgb_pred_tst + 0.30 * cb_pred_tst + 0.20 * xgb_pred_tst), 0, None)

    # ゼロフィルター
    val_stores = train_feat.loc[fam_val_mask, 'store_nbr'].values
    for i, st in enumerate(val_stores):
        if (st, fam) in val_zero_pairs:
            ens_pred_v[i] = 0.0

    tst_stores = test_feat.loc[fam_tst_mask, 'store_nbr'].values
    for i, st in enumerate(tst_stores):
        if (st, fam) in zero_pairs:
            ens_pred_tst[i] = 0.0

    val_ens_preds.extend(ens_pred_v)
    val_trues.extend(y_v)

    fam_test_df = test_feat.loc[fam_tst_mask, ['id']].copy()
    fam_test_df['sales'] = ens_pred_tst
    test_predictions.append(fam_test_df)

# 全体RMSLE計算
val_rmsle = float(np.sqrt(mean_squared_error(np.log1p(val_trues), np.log1p(np.clip(val_ens_preds, 0, None)))))
print('=' * 70)
print(f'🏆 [実験 #006] 特売リード＆ラグ ＋ 3大GBDT 手元検証 RMSLE: {val_rmsle:.5f}')
print(f'  (参考: #004 LGB: 0.40444, #005 アンサンブル: 0.40163)')
print('=' * 70)

## 5. 提出ファイルの生成と確認

In [ ]:
# id順に正確に並べ替え（シャッフルバグ防止）
sub = pd.concat(test_predictions, ignore_index=True)
sub = sub.sort_values('id').reset_index(drop=True)

# 提出CSV保存
sub.to_csv('submission.csv', index=False)
print('submission.csv を出力しました！')
print(f'行数: {len(sub)} (期待値: 28512)')
print(f'欠損値数: {sub.isna().sum().sum()}')
print(f'売上ゼロ件数: {(sub["sales"] == 0).sum()} 行')
print(sub.head(10))